# IOAI — 2026 Summer National Corrupt Codex (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad haio
!cd haio && git sparse-checkout set 2026/nyari-orszagos/feladatok/adatok/korrupt-kodex >/dev/null && git checkout -q
import shutil, glob
for f in glob.glob('haio/2026/nyari-orszagos/feladatok/adatok/korrupt-kodex/*'): shutil.copy(f, '.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Corrupt Codex — 암기(라벨노이즈) 검출 모범답안

HAIO 2026 여름 결선. 일부 라벨이 **오염(flip)** 된 데이터로 학습된 사기탐지 신경망(12→256→256→1, **SIREN**
`sin(30·x)` 활성)은 그 뒤집힌 점들을 **암기(memorize)** 했다. 학습된 가중치(`net_weights.pt`) + 라벨된 보정
50행(`calibration.csv`, is_memorized) + 미라벨 test 5000행으로, **각 test 행이 암기된 오염샘플일 가능성 score** 를
출력한다. **ROC-AUC** 로 채점, 제출 `submission.csv`(id,score).

**핵심 아이디어 — 로짓 되돌림(logit reversion)**: SIREN 은 고주파(`sin30`)라 암기된 뒤집힌 점을 **날카로운 국소
혹(bump)** 으로 만든다. 입력에 **적당한 잡음(σ=0.3)** 을 주면 그 혹은 평균적으로 **주변 매끄러운 장(場)으로
되돌아간다** → `|logit(x) − E[logit(x+noise)]|` 가 암기점에서 크게 나온다. 방향(부호)은 보정 50행으로 맞춘다.

**성능(test 5000, 실측)**: ROC-AUC **≈0.95** (노트북 베이스라인 `-|logit|` 은 0.74). *본질적으로 국소 sharpness
탐지가 로짓 불확실성보다 훨씬 강함.*


In [ ]:
import numpy as np, pandas as pd, torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

# 문제에서 주어진 SIREN 망 (12->256->256->1, sin(30x))
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(12, 256); self.fc2 = nn.Linear(256, 256); self.fc3 = nn.Linear(256, 1)
    def forward(self, x):
        x = torch.sin(30 * self.fc1(x)); x = torch.sin(30 * self.fc2(x)); return self.fc3(x)

model = Net().to(device)
model.load_state_dict(torch.load("net_weights.pt", map_location=device)); model.eval()

cal = pd.read_csv("calibration.csv"); test = pd.read_csv("test.csv")
Xcal = torch.tensor(cal.iloc[:, :12].values, dtype=torch.float32, device=device)
ycal = cal.iloc[:, 12].values                      # is_memorized (0/1), 보정용 50행
Xte  = torch.tensor(test[[f"f{i}" for i in range(12)]].values, dtype=torch.float32, device=device)
print("calibration", len(cal), "test", len(test))


## 로짓 되돌림 신호 + 제출

In [ ]:
# 로짓 되돌림 신호: |logit(x) - E_noise[logit(x+noise)]|  (σ=0.3, K=256)
#   암기된 뒤집힌 점 = 날카로운 국소 혹 → 잡음 평균이 매끄러운 장으로 되돌려 편차가 크다.
SIGMA, K = 0.3, 256
@torch.no_grad()
def reversion_signal(X):
    base = model(X).squeeze(1)
    acc = torch.zeros(len(X), device=device)
    for _ in range(K):
        acc += model(X + SIGMA * torch.randn_like(X)).squeeze(1)
    return (base - acc / K).abs().cpu().numpy()

sig_cal = reversion_signal(Xcal)
sig_te  = reversion_signal(Xte)

# 방향(부호) 보정 — 보정 50행으로 확인(신호가 클수록 암기이면 그대로, 아니면 반전)
cal_auc = roc_auc_score(ycal, sig_cal)
if cal_auc < 0.5:
    sig_cal, sig_te, cal_auc = -sig_cal, -sig_te, 1 - cal_auc
print(f"calibration AUC (logit reversion): {cal_auc:.4f}")

pd.DataFrame({"id": range(len(test)), "score": sig_te}).to_csv("submission.csv", index=False)
print("submission.csv 저장:", len(test), "행")


### 정리
- **로짓 되돌림** `|logit(x) − E[logit(x+noise)]|` (σ=0.3) 로 test ROC-AUC ≈ **0.95** (베이스라인 `-|logit|` 0.74).
- **왜**: SIREN 의 고주파 활성이 암기된 뒤집힌 점을 날카로운 국소 혹으로 만드는데, 적당한 잡음이 그 혹을
  평균적으로 매끄러운 장으로 되돌려 → 암기점에서 편차가 커진다. 방향만 보정 50행으로 맞추면 된다.
- **다른 시도(참고)**: 입력 그래디언트 노름(≈0.48, 무효)·perturb flip-rate(0.68)·로짓 불확실성(0.74) 은 모두
  되돌림보다 약했다. σ 는 0.3~0.4 가 최적(너무 작으면 혹을 못 벗어나고, 너무 크면 신호가 뭉개짐).


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)